<a href="https://colab.research.google.com/github/shyampandey263/langchain-trip-agent/blob/main/langchain_trip_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get -qq install -y zstd pciutils lshw
!curl -fsSL https://ollama.com/install.sh | sh

Selecting previously unselected package pci.ids.
(Reading database ... 126952 files and directories currently installed.)
Preparing to unpack .../0-pci.ids_0.0~2024.03.31-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2024.03.31-1ubuntu0.1) ...
Selecting previously unselected package libpci3:amd64.
Preparing to unpack .../1-libpci3_1%3a3.10.0-2build1_amd64.deb ...
Unpacking libpci3:amd64 (1:3.10.0-2build1) ...
Selecting previously unselected package lshw.
Preparing to unpack .../2-lshw_02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1_amd64.deb ...
Unpacking lshw (02.19.git.2021.06.19.996aaad9c7-2ubuntu0.24.04.1) ...
Selecting previously unselected package pciutils.
Preparing to unpack .../3-pciutils_1%3a3.10.0-2build1_amd64.deb ...
Unpacking pciutils (1:3.10.0-2build1) ...
Selecting previously unselected package usb.ids.
Preparing to unpack .../4-usb.ids_2024.03.18-1_all.deb ...
Unpacking usb.ids (2024.03.18-1) ...
Selecting previously unselected package zstd.
Preparing to unpack .../5-

In [ ]:
import subprocess, time
subprocess.Popen(["ollama", "serve"])
time.sleep(5)
!ollama pull llama3.1

In [ ]:
!nvidia-smi

Mon Sep 21 10:03:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!ollama list

NAME               ID              SIZE      MODIFIED       
llama3.1:latest    46e0c10c039e    4.9 GB    16 minutes ago    


In [ ]:
!pip install -q langchain langchain-ollama

In [ ]:
from langchain.tools import tool

DEMO_PRICES = {"delhi": 95.0, "mumbai": 104.0, "chennai": 101.0}  # made-up demo numbers


@tool
def get_fuel_price(city: str) -> str:
    """Get the petrol price per litre in rupees for a city.
    Use this only when the user asks about the price itself."""
    price = DEMO_PRICES.get(city.lower())
    if price is None:
        return f"No price found for {city}. Available cities: {', '.join(DEMO_PRICES)}."
    return f"Petrol in {city} costs Rs {price} per litre."


@tool
def calculate_trip_cost(city: str, distance_km: float, mileage_km_per_litre: float) -> str:
    """Calculate the fuel cost of a trip in rupees. It looks up the city's petrol price by itself.
    Use this whenever the user asks what a trip will cost."""
    price = DEMO_PRICES.get(city.lower())
    if price is None:
        return f"No price found for {city}. Available cities: {', '.join(DEMO_PRICES)}."
    litres = distance_km / mileage_km_per_litre
    cost = litres * price
    return f"Fuel needed: {litres:.1f} litres at Rs {price} per litre. Total cost: Rs {cost:.0f}."

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

model = init_chat_model("ollama:llama3.1", temperature=0)

agent = create_agent(
    model=model,
    tools=[get_fuel_price, calculate_trip_cost],
    system_prompt=(
        "You are a trip-cost assistant. You cannot do arithmetic yourself. "
        "Step 1: call get_fuel_price for the city and wait for the result. "
        "Step 2: you MUST call calculate_trip_cost with the number from step 1 as price_per_litre. "
        "Never calculate costs in your own text. After the tool returns, state its result."
    ),
)

question = (
    "I will drive 347 km in a car that gives 13.7 km per litre. "
    "Petrol in Mumbai, what will the fuel cost?"
)
result = agent.invoke({"messages": [{"role": "user", "content": question}]})

for message in result["messages"]:
    message.pretty_print()


================================ Human Message =================================

I will drive 347 km in a car that gives 13.7 km per litre. Petrol in Mumbai, what will the fuel cost?
================================== Ai Message ==================================
Tool Calls:
  calculate_trip_cost (722281ac-65cf-46f6-bf03-2f38c9d7465a)
 Call ID: 722281ac-65cf-46f6-bf03-2f38c9d7465a
  Args:
    city: Mumbai
    distance_km: 347
    mileage_km_per_litre: 13.7
================================= Tool Message =================================
Name: calculate_trip_cost

Fuel needed: 25.3 litres at Rs 104.0 per litre. Total cost: Rs 2634.
================================== Ai Message ==================================

The fuel cost for the trip is Rs 2634.
